# Project 11: Adverse Weather Robustness Benchmark (v3.0)

## Obiettivo del progetto

Misurare come degrada un sistema di rilevamento pedoni (baseline YOLOv8n) in CARLA sotto diverse condizioni ambientali e corruzioni visive, identificando **soglie di collasso** e **failure cases**.

### Condizioni testate
- **Meteo**: clear, rain, fog, night, backlight
- **Corruzioni visive**: Gaussian blur, Gaussian noise, motion blur, riduzione contrasto (severità 1-3)
- **Combinazioni**: corruzioni applicate su diverse condizioni meteo

### Metriche raccolte
- Detection rate, first detection distance, min distance
- **Warning delay** (tempo tra ingresso in zona critica e prima detection)
- **Error rate tracking** (miss rate, false positive rate)
- **Forward Collision Warning (FCW)** evaluation

### Deliverables
1. ✅ Benchmark scripts eseguibili (self-contained notebook)
2. ✅ Codice CARLA per gestione scenario e meteo
3. ✅ Report con metodologia, metriche e **failure cases**
4. ✅ Confronti side-by-side visuali
5. ✅ Grafici di degradazione con **soglie di collasso**
6. ✅ Heatmap robustezza weather × corruption
7. ✅ Modulo FCW con valutazione warning

---

## 1. Setup Ambiente e Dipendenze

Prima di iniziare, assicuriamoci di avere tutte le librerie necessarie installate.

**Requisiti**:
- Python 3.7 o 3.8
- CARLA Simulator in esecuzione su `localhost:2000`
- Pacchetti: `carla`, `ultralytics`, `opencv-python`, `pandas`, `matplotlib`, `pillow`, `seaborn`

## 1.1 Fix Dipendenze (se necessario)

Se riscontri errori di import con `urllib3` e OpenSSL, esegui questa cella per downgrade alla versione compatibile.

In [1]:
# Fix compatibilità dipendenze per Python 3.7
import sys
!{sys.executable} -m pip install --upgrade "urllib3<2.0" "typing-extensions>=4.0" "tabulate" "seaborn" --quiet

print("✅ Dipendenze fixate:")
print("   - urllib3 < 2.0 (compatibilità OpenSSL)")
print("   - typing-extensions >= 4.0 (backport typing per Python 3.7)")
print("   - tabulate (per report markdown)")
print("   - seaborn (per heatmap)")

✅ Dipendenze fixate:
   - urllib3 < 2.0 (compatibilità OpenSSL)
   - typing-extensions >= 4.0 (backport typing per Python 3.7)
   - tabulate (per report markdown)
   - seaborn (per heatmap)


In [4]:
%pip install -r requirments.txt

Note: you may need to restart the kernel to use updated packages.


In [5]:
# Verifica versione Python
import sys
print(f"Python version: {sys.version}")

# Fix per typing.OrderedDict in Python 3.7
import typing
if not hasattr(typing, 'OrderedDict'):
    from collections import OrderedDict as CollectionsOrderedDict
    typing.OrderedDict = CollectionsOrderedDict
    print("⚠️ Monkey-patch applicato: typing.OrderedDict aggiunto manualmente")

# Importa librerie principali
import os
import csv
import time
import glob
from datetime import datetime

import carla
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image, ImageFilter, ImageEnhance
from ultralytics import YOLO

sns.set_theme(style="whitegrid")
sns.set_palette("husl")


print("✅ Tutte le librerie importate con successo")

Python version: 3.10.21 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:35:39) [MSC v.1942 64 bit (AMD64)]
✅ Tutte le librerie importate con successo


## 2. Configurazione Connessione CARLA

Connettiamoci al server CARLA e verifichiamo che sia raggiungibile.

**Nota**: Il server CARLA deve essere già avviato prima di eseguire questa cella.

In [6]:
# Configurazione connessione
CARLA_HOST = 'localhost'
CARLA_PORT = 2000
CARLA_TIMEOUT = 10.0

try:
    client = carla.Client(CARLA_HOST, CARLA_PORT)
    client.set_timeout(CARLA_TIMEOUT)
    world = client.get_world()
    
    print(f"✅ Connesso a CARLA")
    print(f"   Client version: {client.get_client_version()}")
    print(f"   Server version: {client.get_server_version()}")
    print(f"   Mappa corrente: {world.get_map().name}")
    
except RuntimeError as e:
    print(f"❌ Errore di connessione: {e}")
    print(f"   Verifica che CARLA sia in esecuzione su {CARLA_HOST}:{CARLA_PORT}")

✅ Connesso a CARLA
   Client version: 0.9.16
   Server version: 0.9.16
   Mappa corrente: Carla/Maps/Town10HD_Opt


## 3. Definizione Condizioni Meteo

Creiamo una funzione per gestire le diverse condizioni meteorologiche in modo consistente.

**Parametri CARLA controllati**:
- Cloudiness, precipitation, fog density, sun position, wetness, wind intensity

In [7]:
def build_weather(weather_type):
    """
    Costruisce parametri meteo CARLA per una condizione specifica.
    
    Args:
        weather_type (str): 'clear', 'rain', 'fog', 'night', 'backlight'
    
    Returns:
        carla.WeatherParameters
    """
    weather_type = weather_type.lower()

    if weather_type == 'clear':
        weather = carla.WeatherParameters.ClearNoon
        weather.cloudiness = 0.0
        weather.precipitation = 0.0
        weather.fog_density = 0.0
        weather.sun_altitude_angle = 70.0
        return weather

    if weather_type == 'rain':
        weather = carla.WeatherParameters.HardRainNoon
        weather.cloudiness = 90.0
        weather.precipitation = 95.0
        weather.fog_density = 8.0
        weather.wetness = 100.0
        return weather

    if weather_type == 'fog':
        weather = carla.WeatherParameters.ClearNoon
        weather.fog_density = 45.0
        weather.fog_distance = 25.0
        weather.fog_falloff = 1.5
        return weather

    if weather_type == 'night':
        weather = carla.WeatherParameters.ClearNight
        weather.sun_altitude_angle = -25.0
        return weather

    if weather_type == 'backlight':
        weather = carla.WeatherParameters.ClearNoon
        weather.sun_azimuth_angle = 180.0
        weather.sun_altitude_angle = 8.0
        return weather

    raise ValueError(f"Condizione meteo sconosciuta: {weather_type}")

# Test rapido
test_weather = build_weather('clear')
print(f"✅ Funzione build_weather() definita e testata")
print(f"   Condizioni supportate: clear, rain, fog, night, backlight")

✅ Funzione build_weather() definita e testata
   Condizioni supportate: clear, rain, fog, night, backlight


## 4. Classe Scenario per Test Ripetibili

Creiamo una classe che gestisce:
- Spawning del veicolo ego
- Spawning del pedone a distanza fissa
- Applicazione condizioni meteo
- Cleanup automatico

In [8]:
class MovingScenario:
    """
    Scenario controllato: ego vehicle si muove verso pedone fermo.
    """
    def __init__(self, world):
        self.world = world
        self.vehicles = []
        self.pedestrians = []
        self.sensors = []
        self.blueprint_lib = world.get_blueprint_library()

    def setup(self, ped_distance=80):
        """
        Crea scenario con pedone a distanza fissa.
        
        Args:
            ped_distance (int): distanza iniziale pedone in metri
        """
        spawn_points = self.world.get_map().get_spawn_points()
        ego_spawn = spawn_points[0]

        # Spawn ego vehicle
        vehicle_bp = self.blueprint_lib.find('vehicle.tesla.model3')
        self.ego_vehicle = self.world.spawn_actor(vehicle_bp, ego_spawn)
        self.vehicles.append(self.ego_vehicle)
        
        # Spawn pedone davanti al veicolo
        ped_location = carla.Location(
            x=ego_spawn.location.x + ped_distance,
            y=ego_spawn.location.y,
            z=ego_spawn.location.z
        )
        pedestrian_bp = self.blueprint_lib.find('walker.pedestrian.0001')
        pedestrian = self.world.spawn_actor(
            pedestrian_bp,
            carla.Transform(ped_location, carla.Rotation(0, 180, 0))
        )
        self.pedestrians.append(pedestrian)
        
        
        # --- BACKGROUND TRAFFIC (1 veicolo) ---
        self.bg_vehicles = []
        bg_vehicle_bp = self.blueprint_lib.find('vehicle.audi.a2')
        opp_location = carla.Location(
            x=ego_spawn.location.x + 120,
            y=ego_spawn.location.y - 4.0,  # Corsia opposta
            z=ego_spawn.location.z + 0.5
        )
        opp_transform = carla.Transform(opp_location, carla.Rotation(0, 180, 0))
        try:
            bg_vehicle = self.world.spawn_actor(bg_vehicle_bp, opp_transform)
            self.bg_vehicles.append(bg_vehicle)
            self.vehicles.append(bg_vehicle)
        except:
            print("Avviso: Impossibile spawnare il veicolo di background.")

        return self.ego_vehicle, self.pedestrians

    def set_weather(self, weather_type):
        """Applica condizione meteo."""
        self.world.set_weather(build_weather(weather_type))

    def cleanup(self):
        """Distruggi tutti gli attori creati."""
        for actor in self.sensors + self.vehicles + self.pedestrians:
            if actor.is_alive:
                actor.destroy()

print("✅ Classe MovingScenario definita")

✅ Classe MovingScenario definita


## 4.1 Scenario 2: Pedone in Attraversamento ← NUOVO

Secondo scenario controllato: il pedone **attraversa la strada** mentre l'ego vehicle si avvicina.

**Perché questo scenario?**
- Più realistico: nella realtà i pedoni attraversano, non stanno fermi
- Più sfidante: il pedone appare da lato, il rilevamento è più difficile
- Rispetta i vincoli della traccia: traiettoria, velocità e traffico rimangono **costanti e fissi**

**Parametri fissi (deterministici)**:
- Punto di attraversamento: sempre 25m avanti dall'ego
- Direzione attraversamento: sempre perpendicolare (da sinistra a destra)
- Velocità pedone: sempre 1.5 m/s
- Traiettoria ego: sempre dritta, throttle costante

In [9]:
class CrossingScenario:
    """
    Scenario 2: pedone attraversa la strada mentre l'ego vehicle si avvicina.
    
    Tutti i parametri sono fissi e deterministici tra un run e l'altro:
    - Posizione di partenza ego: spawn_points[0] (fissa)
    - Punto di attraversamento: CROSSING_DISTANCE avanti dall'ego (fisso)
    - Traiettoria pedone: perpendicolare alla strada, sinistra → destra (fissa)
    - Velocita pedone: PED_SPEED m/s (fissa)
    - Traffico: assente (costante)
    """

    CROSSING_DISTANCE = 25.0  # metri avanti dove il pedone attraversa
    CROSSING_OFFSET = 4.0     # metri a sinistra da cui parte il pedone
    CROSSING_TARGET = 4.0     # metri a destra dove arriva il pedone
    PED_SPEED = 1.5           # m/s — velocita pedonale realistica

    def __init__(self, world):
        self.world = world
        self.vehicles = []
        self.pedestrians = []
        self.controllers = []
        self.sensors = []
        self.blueprint_lib = world.get_blueprint_library()
        self.ego_vehicle = None
        self.pedestrian = None
        self.walker_controller = None

    def setup(self):
        """
        Crea lo scenario: ego vehicle + pedone che attraversa.
        Il pedone inizia a sinistra della strada e cammina verso destra,
        tagliando la traiettoria del veicolo al punto CROSSING_DISTANCE.
        """
        spawn_points = self.world.get_map().get_spawn_points()
        ego_spawn = spawn_points[0]

        # --- EGO VEHICLE ---
        vehicle_bp = self.blueprint_lib.find('vehicle.tesla.model3')
        self.ego_vehicle = self.world.spawn_actor(vehicle_bp, ego_spawn)
        self.vehicles.append(self.ego_vehicle)

        # --- PEDONE: parte a sinistra, cammina verso destra ---
        # Punto di partenza: CROSSING_DISTANCE avanti, CROSSING_OFFSET a sinistra
        ped_start = carla.Location(
            x=ego_spawn.location.x + self.CROSSING_DISTANCE,
            y=ego_spawn.location.y - self.CROSSING_OFFSET,
            z=ego_spawn.location.z
        )
        # Destinazione: stessa X, CROSSING_TARGET a destra
        self.ped_target = carla.Location(
            x=ego_spawn.location.x + self.CROSSING_DISTANCE,
            y=ego_spawn.location.y + self.CROSSING_TARGET,
            z=ego_spawn.location.z
        )

        # Spawna pedone girato verso destra (yaw=90)
        pedestrian_bp = self.blueprint_lib.find('walker.pedestrian.0001')
        ped_transform = carla.Transform(ped_start, carla.Rotation(0, 90, 0))
        self.pedestrian = self.world.spawn_actor(pedestrian_bp, ped_transform)
        self.pedestrians.append(self.pedestrian)

        # --- WALKER AI CONTROLLER ---
        # Richiede un world.tick() prima di avviare
        walker_controller_bp = self.blueprint_lib.find('controller.ai.walker')
        self.walker_controller = self.world.spawn_actor(
            walker_controller_bp,
            carla.Transform(),
            self.pedestrian
        )
        self.controllers.append(self.walker_controller)

        # Tick necessario per inizializzare il controller
        self.world.tick()

        # Avvia il pedone verso la destinazione
        self.walker_controller.start()
        self.walker_controller.go_to_location(self.ped_target)
        self.walker_controller.set_max_speed(self.PED_SPEED)

        print(f"Scenario 2 (Crossing) creato:")
        print(f"   Ego spawn: {ego_spawn.location}")
        print(f"   Pedone parte da: {ped_start} (sinistra della strada)")
        print(f"   Pedone va a: {self.ped_target} (destra della strada)")
        print(f"   Velocita pedone: {self.PED_SPEED} m/s")
        print(f"   Punto attraversamento: {self.CROSSING_DISTANCE}m avanti")

        
        # --- BACKGROUND TRAFFIC (2 veicoli) ---
        self.bg_vehicles = []
        bg_vehicle_bp = self.blueprint_lib.find('vehicle.audi.a2')
        
        opp_location_1 = carla.Location(
            x=ego_spawn.location.x + 100,
            y=ego_spawn.location.y - 4.0,
            z=ego_spawn.location.z + 0.5
        )
        opp_transform_1 = carla.Transform(opp_location_1, carla.Rotation(0, 180, 0))
        try:
            bg_v1 = self.world.spawn_actor(bg_vehicle_bp, opp_transform_1)
            self.bg_vehicles.append(bg_v1)
            self.vehicles.append(bg_v1)
        except:
            pass

        opp_location_2 = carla.Location(
            x=ego_spawn.location.x + 150,
            y=ego_spawn.location.y - 4.0,
            z=ego_spawn.location.z + 0.5
        )
        opp_transform_2 = carla.Transform(opp_location_2, carla.Rotation(0, 180, 0))
        try:
            bg_v2 = self.world.spawn_actor(bg_vehicle_bp, opp_transform_2)
            self.bg_vehicles.append(bg_v2)
            self.vehicles.append(bg_v2)
        except:
            pass

        return self.ego_vehicle, [self.pedestrian]

    def set_weather(self, weather_type):
        """Applica condizione meteo."""
        self.world.set_weather(build_weather(weather_type))

    def cleanup(self):
        """Distruggi tutti gli attori."""
        for ctrl in self.controllers:
            if ctrl.is_alive:
                try:
                    ctrl.stop()
                    ctrl.destroy()
                except:
                    pass
        for actor in self.sensors + self.vehicles + self.pedestrians:
            if actor.is_alive:
                try:
                    actor.destroy()
                except:
                    pass

print("CrossingScenario definito")
print(f"   Punto attraversamento: {CrossingScenario.CROSSING_DISTANCE}m avanti")
print(f"   Velocita pedone: {CrossingScenario.PED_SPEED} m/s (fissa)")
print(f"   Traiettoria: perpendicolare sinistra -> destra (fissa)")

CrossingScenario definito
   Punto attraversamento: 25.0m avanti
   Velocita pedone: 1.5 m/s (fissa)
   Traiettoria: perpendicolare sinistra -> destra (fissa)


## 5. Caricamento Modello YOLOv8 (Baseline)

Utilizziamo YOLOv8n come baseline per il rilevamento pedoni.

**Perché YOLOv8n?**
- Modello leggero e veloce
- Pre-addestrato su COCO (include classe 'person')
- Ideale come baseline per test di robustezza

In [10]:
# Scarica e carica il modello YOLOv8n
MODEL_PATH = 'yolov8n.pt'

try:
    model = YOLO(MODEL_PATH)
    print(f"✅ Modello YOLOv8n caricato da {MODEL_PATH}")
    print(f"   Classi disponibili: {len(model.names)} (inclusa 'person')")
except Exception as e:
    print(f"❌ Errore caricamento modello: {e}")
    print("   Il modello verrà scaricato automaticamente al primo utilizzo")

✅ Modello YOLOv8n caricato da yolov8n.pt
   Classi disponibili: 80 (inclusa 'person')


## 6. Funzioni Utility per Benchmark

Definiamo le funzioni helper per:
- Conversione frame CARLA → RGB numpy
- **Corruzioni visive multiple** (blur, noise, motion blur, contrast) ← NUOVO
- Salvataggio immagini annotate
- Logging risultati CSV

In [11]:
def frame_to_rgb(image):
    """Converte immagine CARLA in array RGB numpy."""
    array = np.frombuffer(image.raw_data, dtype=np.uint8)
    array = array.reshape((image.height, image.width, 4))
    return array[:, :, :3][:, :, ::-1]  # BGRA -> RGB


# ============================================================
# CORRUZIONI VISIVE — Funzioni per degradare le immagini
# ============================================================

def apply_blur(rgb_frame, severity):
    """
    Applica Gaussian blur al frame.
    severity: 1=mild (r=1.5), 2=moderate (r=3.5), 3=severe (r=6.0)
    """
    if severity <= 0:
        return rgb_frame
    radius_map = {1: 1.5, 2: 3.5, 3: 6.0}
    radius = radius_map.get(severity, 3.5)
    pil_image = Image.fromarray(rgb_frame)
    blurred = pil_image.filter(ImageFilter.GaussianBlur(radius=radius))
    return np.array(blurred)


def apply_noise(rgb_frame, severity):
    """
    Applica Gaussian noise al frame.
    severity: 1=mild (sigma=15), 2=moderate (sigma=30), 3=severe (sigma=50)
    """
    if severity <= 0:
        return rgb_frame
    sigma_map = {1: 15, 2: 30, 3: 50}
    sigma = sigma_map.get(severity, 30)
    noise = np.random.normal(0, sigma, rgb_frame.shape).astype(np.float32)
    noisy = np.clip(rgb_frame.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    return noisy


def apply_motion_blur(rgb_frame, severity):
    """
    Applica motion blur orizzontale (simula vibrazioni camera).
    severity: 1=mild (k=3), 2=moderate (k=7)
    """
    if severity <= 0:
        return rgb_frame
    kernel_size_map = {1: 3, 2: 7}
    k = kernel_size_map.get(severity, 5)
    
    # Kernel di motion blur orizzontale
    kernel_1d = [1.0 / k] * k
    pil_img = Image.fromarray(rgb_frame)
    blurred = pil_img.filter(ImageFilter.Kernel(
        size=(k, 1),
        kernel=kernel_1d,
        scale=1,
        offset=0
    ))
    return np.array(blurred)


def apply_contrast_reduction(rgb_frame, severity):
    """
    Riduce il contrasto dell'immagine.
    severity: 1=mild (0.7), 2=moderate (0.4)
    """
    if severity <= 0:
        return rgb_frame
    factor_map = {1: 0.7, 2: 0.4}
    factor = factor_map.get(severity, 0.4)
    pil_image = Image.fromarray(rgb_frame)
    enhancer = ImageEnhance.Contrast(pil_image)
    reduced = enhancer.enhance(factor)
    return np.array(reduced)


def apply_corruption(rgb_frame, corruption_type, severity):
    """
    Dispatcher: applica la corruzione specificata al frame.
    
    Args:
        rgb_frame: numpy array RGB
        corruption_type: 'none', 'blur', 'noise', 'motion_blur', 'contrast'
        severity: livello di severita (0 = nessuna)
    
    Returns:
        numpy array RGB con corruzione applicata
    """
    if corruption_type == 'none' or severity <= 0:
        return rgb_frame
    
    corruption_functions = {
        'blur': apply_blur,
        'noise': apply_noise,
        'motion_blur': apply_motion_blur,
        'contrast': apply_contrast_reduction,
    }
    
    fn = corruption_functions.get(corruption_type)
    if fn is None:
        raise ValueError(f"Corruzione sconosciuta: {corruption_type}")
    
    return fn(rgb_frame, severity)


def save_annotated_sample(result, output_path):
    """Salva frame con bounding box YOLO."""
    annotated = result.plot()
    annotated_rgb = annotated[:, :, ::-1]
    Image.fromarray(annotated_rgb).save(output_path)


def append_csv_rows(path, fieldnames, rows):
    """Aggiunge righe a CSV (crea file se non esiste)."""
    if not rows:
        return
    
    file_exists = os.path.exists(path)
    with open(path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)


print("✅ Funzioni utility definite")
print("   Corruzioni disponibili: blur, noise, motion_blur, contrast")
print("   Livelli severita: 1 (mild), 2 (moderate), 3 (severe)")

✅ Funzioni utility definite
   Corruzioni disponibili: blur, noise, motion_blur, contrast
   Livelli severita: 1 (mild), 2 (moderate), 3 (severe)


## 7. Configurazione Test Batch

Definiamo le condizioni da testare. Rispetto alla v2.0, aggiungiamo:
- **Gaussian noise** (severità 1, 2, 3) — simula sensori di bassa qualità
- **Motion blur** (severità 1, 2) — simula vibrazioni della camera
- **Riduzione contrasto** (severità 1, 2) — simula condizioni di scarsa illuminazione
- **Combinazioni** weather + corruption per test incrociati

Totale: **17 condizioni** (vs. 7 della v2.0)

In [12]:
# Parametri simulazione
WIDTH, HEIGHT = 800, 600
FOV = 90
PEDESTRIAN_DISTANCE_M = 80
WARNING_DISTANCE_M = 15.0
MAX_FRAMES_PER_RUN = 450
THROTTLE = 0.4
FRAME_INTERVAL_S = 0.03

# ============================================================
# CONDIZIONI DA TESTARE (17 totali)
# ============================================================
TESTS = [
    # --- Condizioni meteo (baseline: nessuna corruzione) ---
    {'weather': 'clear',     'corruption': 'none',     'severity': 0},
    {'weather': 'rain',      'corruption': 'none',     'severity': 0},
    {'weather': 'fog',       'corruption': 'none',     'severity': 0},
    {'weather': 'night',     'corruption': 'none',     'severity': 0},
    {'weather': 'backlight', 'corruption': 'none',     'severity': 0},
    
    # --- Blur progressivo (clear weather) ---
    {'weather': 'clear', 'corruption': 'blur', 'severity': 1},
    {'weather': 'clear', 'corruption': 'blur', 'severity': 2},
    {'weather': 'clear', 'corruption': 'blur', 'severity': 3},
    
    # --- Gaussian noise progressivo (clear weather) ---
    {'weather': 'clear', 'corruption': 'noise', 'severity': 1},
    {'weather': 'clear', 'corruption': 'noise', 'severity': 2},
    {'weather': 'clear', 'corruption': 'noise', 'severity': 3},
    
    # --- Motion blur (clear weather) ---
    {'weather': 'clear', 'corruption': 'motion_blur', 'severity': 1},
    {'weather': 'clear', 'corruption': 'motion_blur', 'severity': 2},
    
    # --- Riduzione contrasto (clear weather) ---
    {'weather': 'clear', 'corruption': 'contrast', 'severity': 1},
    {'weather': 'clear', 'corruption': 'contrast', 'severity': 2},
    
    # --- Combinazioni critiche (weather + corruption) ---
    {'weather': 'rain', 'corruption': 'blur',  'severity': 1},
    {'weather': 'fog',  'corruption': 'noise', 'severity': 1},
]

# Path output - NUOVA CARTELLA CON TIMESTAMP
batch_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
BASE_OUTPUT_DIR = f'output_batch_{batch_timestamp}'
RESULTS_DIR = os.path.join(BASE_OUTPUT_DIR, 'results')
SAMPLES_DIR = os.path.join(BASE_OUTPUT_DIR, 'detection_samples')
PLOTS_DIR = os.path.join(BASE_OUTPUT_DIR, 'plots')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

FRAME_LOG_PATH = os.path.join(RESULTS_DIR, 'frame_metrics.csv')
RUN_SUMMARY_PATH = os.path.join(RESULTS_DIR, 'run_summary.csv')

print(f"✅ Configurazione benchmark pronta")
print(f"   Test da eseguire: {len(TESTS)}")
print(f"   Nuova cartella output: {BASE_OUTPUT_DIR}")
print(f"   Sample detection: {SAMPLES_DIR}")
print(f"   Risultati CSV: {RESULTS_DIR}")
print()
print("Test configurati:")
for i, t in enumerate(TESTS, 1):
    print(f"   [{i:2d}] weather={t['weather']:10s} corruption={t['corruption']:12s} severity={t['severity']}")

✅ Configurazione benchmark pronta
   Test da eseguire: 17
   Nuova cartella output: output_batch_20260925_145944
   Sample detection: output_batch_20260925_145944\detection_samples
   Risultati CSV: output_batch_20260925_145944\results

Test configurati:
   [ 1] weather=clear      corruption=none         severity=0
   [ 2] weather=rain       corruption=none         severity=0
   [ 3] weather=fog        corruption=none         severity=0
   [ 4] weather=night      corruption=none         severity=0
   [ 5] weather=backlight  corruption=none         severity=0
   [ 6] weather=clear      corruption=blur         severity=1
   [ 7] weather=clear      corruption=blur         severity=2
   [ 8] weather=clear      corruption=blur         severity=3
   [ 9] weather=clear      corruption=noise        severity=1
   [10] weather=clear      corruption=noise        severity=2
   [11] weather=clear      corruption=noise        severity=3
   [12] weather=clear      corruption=motion_blur  severity=1
 

## 8. Funzione Esecuzione Singola Condizione

Core del benchmark: esegue un test completo per una combinazione meteo+corruzione.

**Metriche raccolte per ogni run**:
- `detection_rate`: percentuale frame con pedone rilevato (conf ≥ 0.6)
- `first_detection_distance_m`: distanza alla prima detection
- `min_distance_m`: distanza minima veicolo-pedone raggiunta
- `warning_delay_s`: tempo tra ingresso in zona warning (< 15m) e prima detection ← **RIPRISTINATO**
- `miss_rate`: percentuale di frame in zona critica (< 30m) senza detection ← **NUOVO**
- `false_positive_count`: detection "person" in frame dove il pedone è troppo lontano (> 60m) ← **NUOVO**

### Soglia di confidenza: 0.6

In condizioni avverse YOLO rileva erroneamente oggetti come "person" con confidenza bassa (~0.53).
Soglia aumentata a **0.6** per ridurre i falsi positivi.

In [13]:
def run_condition(world, model, batch_id, weather, corruption, severity):
    """
    Esegue un test completo per una condizione specifica.
    
    Metriche raccolte:
    - detection_rate, first_detection_distance, min_distance
    - warning_delay_s (tempo tra zona warning e prima detection)
    - miss_rate (frame in zona critica senza detection)
    - false_positive_count (detection dove pedone e' lontano)
    
    Returns:
        tuple: (summary_row, frame_rows)
    """
    run_label = f"{weather}_{corruption}{severity}"
    print(f"\n{'='*50}")
    print(f"  Running: {run_label}")
    print(f"{'='*50}")

    # Soglia di confidenza aumentata per ridurre falsi positivi
    CONFIDENCE_THRESHOLD = 0.6
    # Zona critica: distanza sotto cui il pedone DOVREBBE essere rilevato
    CRITICAL_DISTANCE_M = 30.0
    # Distanza oltre la quale una detection "person" e' probabilmente un falso positivo
    FAR_DISTANCE_M = 60.0

    frame_rows = []
    summary_row = {
        'batch_id': batch_id,
        'run_label': run_label,
        'weather': weather,
        'corruption': corruption,
        'severity': severity,
        'total_frames': 0,
        'detection_frames': 0,
        'detection_rate': 0.0,
        'first_detection_distance_m': None,
        'first_detection_elapsed_s': None,
        'warning_delay_s': None,
        'min_distance_m': None,
        'miss_rate': 0.0,
        'false_positive_count': 0,
        'critical_zone_frames': 0,
        'critical_zone_misses': 0,
        'status': 'ok',
    }

    scenario = None
    camera = None
    latest_frame = {'array': None}
    first_detection_distance = None
    first_detection_time = None
    entered_warning_time = None
    frame_count = 0
    detection_count = 0
    min_distance = None
    detection_classes = {}
    low_conf_detections = 0
    false_positive_count = 0
    critical_zone_frames = 0
    critical_zone_misses = 0
    run_start_time = time.perf_counter()
    
    # Sotto-cartella per sample di questa condizione
    condition_samples_dir = os.path.join(SAMPLES_DIR, run_label)
    os.makedirs(condition_samples_dir, exist_ok=True)
    samples_saved = 0

    try:
        # Setup scenario
        scenario = MovingScenario(world)
        scenario.setup(ped_distance=PEDESTRIAN_DISTANCE_M)
        scenario.set_weather(weather)

        # Attach camera
        camera_bp = world.get_blueprint_library().find('sensor.camera.rgb')
        camera_bp.set_attribute('image_size_x', str(WIDTH))
        camera_bp.set_attribute('image_size_y', str(HEIGHT))
        camera_bp.set_attribute('fov', str(FOV))
        camera_transform = carla.Transform(carla.Location(x=1.5, z=2.4))
        camera = world.spawn_actor(camera_bp, camera_transform, attach_to=scenario.ego_vehicle)

        def on_image(image):
            latest_frame['array'] = frame_to_rgb(image)

        camera.listen(on_image)
        scenario.ego_vehicle.set_autopilot(False)

        # Main loop
        while frame_count < MAX_FRAMES_PER_RUN:
            scenario.ego_vehicle.apply_control(carla.VehicleControl(throttle=THROTTLE, steer=0.0))
            if hasattr(scenario, 'bg_vehicles'):
                for bg_v in scenario.bg_vehicles:
                    bg_v.apply_control(carla.VehicleControl(throttle=THROTTLE, steer=0.0))
            world.tick()
            time.sleep(FRAME_INTERVAL_S)

            if latest_frame['array'] is None:
                continue

            frame_count += 1
            current_frame = latest_frame['array']
            
            # Applica corruzione visiva (dispatcher generico)
            processed_frame = apply_corruption(current_frame, corruption, severity)

            # Calcola distanza veicolo-pedone
            v_loc = scenario.ego_vehicle.get_location()
            p_loc = scenario.pedestrians[0].get_location()
            distance = v_loc.distance(p_loc)
            min_distance = distance if min_distance is None else min(min_distance, distance)
            elapsed_s = time.perf_counter() - run_start_time

            # Tracking zona warning (< 15m)
            if distance <= WARNING_DISTANCE_M and entered_warning_time is None:
                entered_warning_time = time.perf_counter()

            # Tracking zona critica (< 30m) per miss rate
            in_critical_zone = distance <= CRITICAL_DISTANCE_M
            if in_critical_zone:
                critical_zone_frames += 1

            # YOLO detection
            results = model(processed_frame, verbose=False)
            result = results[0]

            detected = False
            person_boxes = []
            max_confidence = 0.0
            persons_detected = 0
            
            for box in result.boxes:
                cls_name = model.names[int(box.cls[0])]
                conf = float(box.conf[0])
                
                # Track tutte le classi rilevate
                if cls_name not in detection_classes:
                    detection_classes[cls_name] = 0
                detection_classes[cls_name] += 1
                
                if cls_name == 'person':
                    if conf >= CONFIDENCE_THRESHOLD:
                        detected = True
                        persons_detected += 1
                        max_confidence = max(max_confidence, conf)
                        person_boxes.append((conf, box))
                    else:
                        low_conf_detections += 1
            
            # Error rate tracking
            if detected and distance > FAR_DISTANCE_M:
                false_positive_count += 1
            
            if in_critical_zone and not detected:
                critical_zone_misses += 1

            if detected:
                detection_count += 1
                if first_detection_distance is None:
                    first_detection_distance = distance
                    first_detection_time = time.perf_counter()
                    print(f"  Prima detection PERSON: {distance:.1f}m (conf={person_boxes[0][0]:.2f})")

            # Frame-level row per CSV
            frame_rows.append({
                'batch_id': batch_id,
                'run_label': run_label,
                'weather': weather,
                'corruption': corruption,
                'severity': severity,
                'frame_index': frame_count,
                'elapsed_s': round(elapsed_s, 3),
                'distance_m': round(distance, 3),
                'persons_detected': persons_detected,
                'max_confidence': round(max_confidence, 4),
                'detected': int(detected),
                'in_critical_zone': int(in_critical_zone),
            })

            # Salva sample a intervalli regolari o quando c'e' detection
            save_sample = False
            sample_label = ""
            
            if frame_count == 1:
                save_sample = True
                sample_label = f"frame001_start_{distance:.1f}m"
            elif detected and samples_saved < 5:
                save_sample = True
                sample_label = f"frame{frame_count:03d}_detected_{distance:.1f}m"
            elif in_critical_zone and not detected and samples_saved < 8:
                # NUOVO: salva failure case (miss in zona critica)
                save_sample = True
                sample_label = f"frame{frame_count:03d}_MISS_{distance:.1f}m"
            elif frame_count % 50 == 0 and samples_saved < 10:
                save_sample = True
                sample_label = f"frame{frame_count:03d}_interval_{distance:.1f}m"
            
            if save_sample:
                sample_path = os.path.join(condition_samples_dir, f"{sample_label}.png")
                save_annotated_sample(result, sample_path)
                samples_saved += 1

            # Stop se troppo vicino
            if distance < 3.0:
                last_path = os.path.join(condition_samples_dir, f"frame{frame_count:03d}_final_{distance:.1f}m.png")
                save_annotated_sample(result, last_path)
                print('  Veicolo troppo vicino, stop.')
                break

        # Calcola metriche finali
        detection_rate = detection_count / frame_count if frame_count else 0.0
        miss_rate = critical_zone_misses / critical_zone_frames if critical_zone_frames else 0.0
        
        # Warning delay: tempo tra ingresso in zona warning e prima detection
        warning_delay = None
        if first_detection_time is not None and entered_warning_time is not None:
            warning_delay = round(max(0.0, first_detection_time - entered_warning_time), 3)
        
        first_detection_elapsed = None
        if first_detection_time is not None:
            first_detection_elapsed = round(first_detection_time - run_start_time, 3)
        
        summary_row.update({
            'total_frames': frame_count,
            'detection_frames': detection_count,
            'detection_rate': round(detection_rate, 4),
            'first_detection_distance_m': round(first_detection_distance, 3) if first_detection_distance else None,
            'first_detection_elapsed_s': first_detection_elapsed,
            'warning_delay_s': warning_delay,
            'min_distance_m': round(min_distance, 3) if min_distance else None,
            'miss_rate': round(miss_rate, 4),
            'false_positive_count': false_positive_count,
            'critical_zone_frames': critical_zone_frames,
            'critical_zone_misses': critical_zone_misses,
        })

        print(f"  Risultato {run_label}: detection_rate={summary_row['detection_rate']:.2%}")
        print(f"     Sample salvati: {samples_saved}")
        print(f"     Classi rilevate: {detection_classes}")
        print(f"     Warning delay: {warning_delay}s")
        print(f"     Miss rate (zona critica): {miss_rate:.2%} ({critical_zone_misses}/{critical_zone_frames})")
        if false_positive_count > 0:
            print(f"     Falsi positivi (person a >60m): {false_positive_count}")
        if low_conf_detections > 0:
            print(f"     Detection person scartate (conf<{CONFIDENCE_THRESHOLD}): {low_conf_detections}")

    except Exception as exc:
        summary_row['status'] = 'failed'
        print(f"  ERRORE {run_label}: {exc}")

    finally:
        if camera is not None:
            try:
                camera.stop()
                camera.destroy()
            except:
                pass
        if scenario is not None:
            scenario.cleanup()

    return summary_row, frame_rows

print("Funzione run_condition() definita")
print("   Soglia confidenza PERSON: 0.6")
print("   Metriche: detection_rate, warning_delay, miss_rate, false_positive_count")

Funzione run_condition() definita
   Soglia confidenza PERSON: 0.6
   Metriche: detection_rate, warning_delay, miss_rate, false_positive_count


## 8.1 Forward Collision Warning (FCW) Module

Implementiamo un sistema di **Forward Collision Warning** basilare che:
1. Usa la detection YOLOv8 per identificare pedoni
2. Genera un **WARNING** quando il pedone è rilevato entro una soglia critica (20m)
3. Genera un **DANGER** quando il pedone è rilevato entro 10m
4. Traccia le performance del warning system:
   - **Precision**: quanti warning sono corretti (vs falsi allarmi)
   - **Recall**: quanti pericoli reali vengono segnalati (vs missed warnings)
   - **Warning delay**: tempo tra l'ingresso nella zona pericolosa e il primo warning

In [14]:
class ForwardCollisionWarning:
    """
    Modulo FCW basilare basato su detection YOLOv8.
    
    Logica:
    - Se detection 'person' con conf >= threshold E distanza < FCW_DISTANCE -> WARNING
    - Se distanza < DANGER_DISTANCE -> DANGER
    - Traccia true positives, missed warnings, false alarms
    """
    
    FCW_WARNING_DISTANCE = 20.0  # metri: soglia per emettere warning
    FCW_DANGER_DISTANCE = 10.0   # metri: soglia pericolo critico
    
    def __init__(self):
        self.warnings_issued = 0
        self.true_positive_warnings = 0   # Warning corretto (pedone vicino)
        self.missed_warnings = 0           # Pedone vicino, nessun warning
        self.false_alarms = 0              # Warning ma pedone lontano
        self.first_warning_time = None
        self.first_warning_distance = None
        self.danger_zone_entered_time = None
        self.warning_log = []
    
    def evaluate_frame(self, detected, distance, confidence, elapsed_s):
        """
        Valuta un singolo frame per FCW.
        
        Args:
            detected (bool): se YOLO ha rilevato 'person'
            distance (float): distanza reale veicolo-pedone
            confidence (float): confidenza massima detection
            elapsed_s (float): tempo trascorso dall'inizio del run
        
        Returns:
            dict: stato warning per questo frame
        """
        in_warning_zone = distance <= self.FCW_WARNING_DISTANCE
        in_danger_zone = distance <= self.FCW_DANGER_DISTANCE
        
        # Track quando si entra nella zona di pericolo
        if in_danger_zone and self.danger_zone_entered_time is None:
            self.danger_zone_entered_time = elapsed_s
        
        warning_issued = False
        warning_type = 'none'
        
        if detected and in_warning_zone:
            # TRUE POSITIVE: detection corretta in zona warning
            warning_issued = True
            warning_type = 'DANGER' if in_danger_zone else 'WARNING'
            self.true_positive_warnings += 1
            self.warnings_issued += 1
            
            if self.first_warning_time is None:
                self.first_warning_time = elapsed_s
                self.first_warning_distance = distance
                
        elif not detected and in_warning_zone:
            # MISSED WARNING: pedone vicino ma non rilevato
            self.missed_warnings += 1
            warning_type = 'MISSED'
            
        elif detected and not in_warning_zone:
            if distance > 60.0:
                self.false_alarms += 1
                warning_type = 'FALSE_ALARM'
        
        frame_result = {
            'warning_issued': warning_issued,
            'warning_type': warning_type,
            'distance_m': distance,
            'in_warning_zone': in_warning_zone,
            'in_danger_zone': in_danger_zone,
        }
        self.warning_log.append(frame_result)
        return frame_result
    
    def get_summary(self):
        """Restituisce metriche aggregate FCW."""
        total_warning_zone_frames = sum(1 for f in self.warning_log if f['in_warning_zone'])
        total_danger_zone_frames = sum(1 for f in self.warning_log if f['in_danger_zone'])
        
        fcw_precision = (self.true_positive_warnings / 
                        (self.true_positive_warnings + self.false_alarms)
                        if (self.true_positive_warnings + self.false_alarms) > 0 else 0.0)
        
        fcw_recall = (self.true_positive_warnings / 
                     (self.true_positive_warnings + self.missed_warnings)
                     if (self.true_positive_warnings + self.missed_warnings) > 0 else 0.0)
        
        warning_delay = None
        if self.first_warning_time is not None and self.danger_zone_entered_time is not None:
            warning_delay = max(0.0, self.first_warning_time - self.danger_zone_entered_time)
        
        return {
            'fcw_warnings_issued': self.warnings_issued,
            'fcw_true_positives': self.true_positive_warnings,
            'fcw_missed_warnings': self.missed_warnings,
            'fcw_false_alarms': self.false_alarms,
            'fcw_precision': round(fcw_precision, 4),
            'fcw_recall': round(fcw_recall, 4),
            'fcw_first_warning_distance_m': round(self.first_warning_distance, 2) if self.first_warning_distance else None,
            'fcw_warning_delay_s': round(warning_delay, 3) if warning_delay is not None else None,
            'fcw_warning_zone_frames': total_warning_zone_frames,
            'fcw_danger_zone_frames': total_danger_zone_frames,
        }

print("Modulo FCW definito")
print(f"   Soglia WARNING: {ForwardCollisionWarning.FCW_WARNING_DISTANCE}m")
print(f"   Soglia DANGER: {ForwardCollisionWarning.FCW_DANGER_DISTANCE}m")

Modulo FCW definito
   Soglia WARNING: 20.0m
   Soglia DANGER: 10.0m


## 9. Esecuzione Benchmark Completo

⚠️ **ATTENZIONE**: Questa cella esegue tutti i 17 test configurati. Può richiedere diversi minuti.

Per ogni condizione viene anche valutato il **modulo FCW**.

In [15]:
# Genera batch ID univoco
batch_id = datetime.now().strftime('%Y%m%d_%H%M%S')
all_summary_rows = []
all_frame_rows = []
fcw_results = {}

print(f"Benchmark avviato | batch={batch_id}")
print(f"   Test totali: {len(TESTS)}")
print("-" * 60)

for idx, test in enumerate(TESTS, 1):
    print(f"\n[{idx}/{len(TESTS)}] Test in corso...")
    
    summary_row, frame_rows = run_condition(
        world=world,
        model=model,
        batch_id=batch_id,
        weather=test['weather'],
        corruption=test['corruption'],
        severity=test['severity'],
    )
    
    # Valutazione FCW su frame_rows
    fcw = ForwardCollisionWarning()
    for fr in frame_rows:
        fcw.evaluate_frame(
            detected=bool(fr['detected']),
            distance=fr['distance_m'],
            confidence=fr['max_confidence'],
            elapsed_s=fr['elapsed_s'],
        )
    
    fcw_summary = fcw.get_summary()
    summary_row.update(fcw_summary)
    fcw_results[summary_row['run_label']] = fcw_summary
    
    all_summary_rows.append(summary_row)
    all_frame_rows.extend(frame_rows)

# Salva risultati
SUMMARY_FIELDS = [
    'batch_id', 'run_label', 'weather', 'corruption', 'severity',
    'total_frames', 'detection_frames', 'detection_rate',
    'first_detection_distance_m', 'first_detection_elapsed_s',
    'warning_delay_s', 'min_distance_m',
    'miss_rate', 'false_positive_count',
    'critical_zone_frames', 'critical_zone_misses',
    'fcw_warnings_issued', 'fcw_true_positives', 'fcw_missed_warnings',
    'fcw_false_alarms', 'fcw_precision', 'fcw_recall',
    'fcw_first_warning_distance_m', 'fcw_warning_delay_s',
    'status',
]

FRAME_FIELDS = [
    'batch_id', 'run_label', 'weather', 'corruption', 'severity',
    'frame_index', 'elapsed_s', 'distance_m',
    'persons_detected', 'max_confidence', 'detected', 'in_critical_zone',
]

append_csv_rows(RUN_SUMMARY_PATH, SUMMARY_FIELDS, all_summary_rows)
append_csv_rows(FRAME_LOG_PATH, FRAME_FIELDS, all_frame_rows)

print("\n" + "=" * 60)
print(f"Benchmark completato!")
print(f"   Risultati salvati: {RUN_SUMMARY_PATH}")
print(f"   Frame log salvato: {FRAME_LOG_PATH}")
print(f"   Sample annotate: {SAMPLES_DIR}")

Benchmark avviato | batch=20260925_145959
   Test totali: 17
------------------------------------------------------------

[1/17] Test in corso...

  Running: clear_none0
  Prima detection PERSON: 14.1m (conf=0.70)
  Veicolo troppo vicino, stop.
  Risultato clear_none0: detection_rate=10.53%
     Sample salvati: 9
     Classi rilevate: {'car': 39, 'traffic light': 79, 'person': 32, 'bus': 1, 'fire hydrant': 1}
     Warning delay: 0.228s
     Miss rate (zona critica): 60.00% (9/15)
     Detection person scartate (conf<0.6): 26

[2/17] Test in corso...

  Running: rain_none0
  Prima detection PERSON: 18.3m (conf=0.62)
  Veicolo troppo vicino, stop.
  Risultato rain_none0: detection_rate=16.90%
     Sample salvati: 8
     Classi rilevate: {'traffic light': 105, 'car': 9, 'person': 38, 'train': 1, 'bench': 1}
     Warning delay: 0.0s
     Miss rate (zona critica): 33.33% (6/18)
     Detection person scartate (conf<0.6): 26

[3/17] Test in corso...

  Running: fog_none0
  Prima detection PE

KeyboardInterrupt: 

## 9.2 Esecuzione Scenario 2: Pedone in Attraversamento ← NUOVO

Eseguiamo un **subset di condizioni** (6 vs 17) per Scenario 2, focalizzandoci sulle condizioni più significative.

I risultati verranno poi confrontati con Scenario 1 nella sezione di analisi comparativa.

In [ ]:
# ============================================================
# SCENARIO 2 — PEDONE CHE ATTRAVERSA
# Subset di 6 condizioni chiave
# ============================================================

TESTS_SCENARIO2 = [
    {'weather': 'clear',  'corruption': 'none', 'severity': 0},   # baseline
    {'weather': 'rain',   'corruption': 'none', 'severity': 0},   # peggiore meteo
    {'weather': 'fog',    'corruption': 'none', 'severity': 0},   # visibilita ridotta
    {'weather': 'night',  'corruption': 'none', 'severity': 0},   # buio
    {'weather': 'clear',  'corruption': 'blur', 'severity': 2},   # corruzione moderata
    {'weather': 'clear',  'corruption': 'blur', 'severity': 3},   # corruzione severa
]

# Path output separato per Scenario 2
S2_OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, 'scenario2')
S2_RESULTS_DIR = os.path.join(S2_OUTPUT_DIR, 'results')
S2_SAMPLES_DIR = os.path.join(S2_OUTPUT_DIR, 'detection_samples')
S2_PLOTS_DIR = os.path.join(S2_OUTPUT_DIR, 'plots')

os.makedirs(S2_RESULTS_DIR, exist_ok=True)
os.makedirs(S2_SAMPLES_DIR, exist_ok=True)
os.makedirs(S2_PLOTS_DIR, exist_ok=True)

S2_SUMMARY_PATH = os.path.join(S2_RESULTS_DIR, 'run_summary_s2.csv')
S2_FRAME_PATH = os.path.join(S2_RESULTS_DIR, 'frame_metrics_s2.csv')

# ---- Funzione run_condition adattata per CrossingScenario ----
def run_condition_crossing(world, model, batch_id, weather, corruption, severity):
    """
    Variante di run_condition che usa CrossingScenario (pedone in attraversamento).
    Usa lo stesso SAMPLES_DIR globale ma con prefisso 's2_' nel run_label.
    """
    run_label = f"s2_{weather}_{corruption}{severity}"
    print(f"\n{'='*50}")
    print(f"  [Scenario 2] Running: {run_label}")
    print(f"{'='*50}")

    CONFIDENCE_THRESHOLD = 0.6
    CRITICAL_DISTANCE_M = 20.0   # Soglia critica ridotta: il pedone attraversa vicino
    FAR_DISTANCE_M = 50.0

    frame_rows = []
    summary_row = {
        'batch_id': batch_id,
        'run_label': run_label,
        'scenario': 'crossing',
        'weather': weather,
        'corruption': corruption,
        'severity': severity,
        'total_frames': 0,
        'detection_frames': 0,
        'detection_rate': 0.0,
        'first_detection_distance_m': None,
        'warning_delay_s': None,
        'min_distance_m': None,
        'miss_rate': 0.0,
        'false_positive_count': 0,
        'critical_zone_frames': 0,
        'critical_zone_misses': 0,
        'status': 'ok',
    }

    scenario = None
    camera = None
    latest_frame = {'array': None}
    first_detection_distance = None
    first_detection_time = None
    entered_warning_time = None
    frame_count = 0
    detection_count = 0
    min_distance = None
    false_positive_count = 0
    critical_zone_frames = 0
    critical_zone_misses = 0
    run_start_time = time.perf_counter()

    condition_samples_dir = os.path.join(S2_SAMPLES_DIR, run_label)
    os.makedirs(condition_samples_dir, exist_ok=True)
    samples_saved = 0

    try:
        # Usa CrossingScenario invece di MovingScenario
        scenario = CrossingScenario(world)
        scenario.setup()
        scenario.set_weather(weather)

        camera_bp = world.get_blueprint_library().find('sensor.camera.rgb')
        camera_bp.set_attribute('image_size_x', str(WIDTH))
        camera_bp.set_attribute('image_size_y', str(HEIGHT))
        camera_bp.set_attribute('fov', str(FOV))
        camera_transform = carla.Transform(carla.Location(x=1.5, z=2.4))
        camera = world.spawn_actor(camera_bp, camera_transform,
                                   attach_to=scenario.ego_vehicle)

        def on_image(image):
            latest_frame['array'] = frame_to_rgb(image)

        camera.listen(on_image)
        scenario.ego_vehicle.set_autopilot(False)

        while frame_count < MAX_FRAMES_PER_RUN:
            scenario.ego_vehicle.apply_control(
                carla.VehicleControl(throttle=THROTTLE, steer=0.0))
            if hasattr(scenario, 'bg_vehicles'):
                for bg_v in scenario.bg_vehicles:
                    bg_v.apply_control(carla.VehicleControl(throttle=THROTTLE, steer=0.0))
            world.tick()
            time.sleep(FRAME_INTERVAL_S)

            if latest_frame['array'] is None:
                continue

            frame_count += 1
            current_frame = latest_frame['array']
            processed_frame = apply_corruption(current_frame, corruption, severity)

            v_loc = scenario.ego_vehicle.get_location()
            p_loc = scenario.pedestrian.get_location()
            distance = v_loc.distance(p_loc)
            min_distance = distance if min_distance is None else min(min_distance, distance)
            elapsed_s = time.perf_counter() - run_start_time

            if distance <= WARNING_DISTANCE_M and entered_warning_time is None:
                entered_warning_time = time.perf_counter()

            in_critical_zone = distance <= CRITICAL_DISTANCE_M
            if in_critical_zone:
                critical_zone_frames += 1

            results = model(processed_frame, verbose=False)
            result = results[0]

            detected = False
            max_confidence = 0.0
            persons_detected = 0

            for box in result.boxes:
                cls_name = model.names[int(box.cls[0])]
                conf = float(box.conf[0])
                if cls_name == 'person' and conf >= CONFIDENCE_THRESHOLD:
                    detected = True
                    persons_detected += 1
                    max_confidence = max(max_confidence, conf)

            if detected and distance > FAR_DISTANCE_M:
                false_positive_count += 1
            if in_critical_zone and not detected:
                critical_zone_misses += 1

            if detected:
                detection_count += 1
                if first_detection_distance is None:
                    first_detection_distance = distance
                    first_detection_time = time.perf_counter()
                    print(f"  Prima detection: {distance:.1f}m (conf={max_confidence:.2f})")

            frame_rows.append({
                'batch_id': batch_id,
                'run_label': run_label,
                'scenario': 'crossing',
                'weather': weather,
                'corruption': corruption,
                'severity': severity,
                'frame_index': frame_count,
                'elapsed_s': round(elapsed_s, 3),
                'distance_m': round(distance, 3),
                'persons_detected': persons_detected,
                'max_confidence': round(max_confidence, 4),
                'detected': int(detected),
                'in_critical_zone': int(in_critical_zone),
            })

            # Salva sample
            if frame_count == 1:
                save_annotated_sample(result,
                    os.path.join(condition_samples_dir, f"frame001_start_{distance:.1f}m.png"))
                samples_saved += 1
            elif detected and samples_saved < 5:
                save_annotated_sample(result,
                    os.path.join(condition_samples_dir,
                                 f"frame{frame_count:03d}_detected_{distance:.1f}m.png"))
                samples_saved += 1
            elif in_critical_zone and not detected and samples_saved < 8:
                save_annotated_sample(result,
                    os.path.join(condition_samples_dir,
                                 f"frame{frame_count:03d}_MISS_{distance:.1f}m.png"))
                samples_saved += 1

            if distance < 2.0:
                print('  Veicolo molto vicino al punto di attraversamento, stop.')
                break

        detection_rate = detection_count / frame_count if frame_count else 0.0
        miss_rate = critical_zone_misses / critical_zone_frames if critical_zone_frames else 0.0
        warning_delay = None
        if first_detection_time is not None and entered_warning_time is not None:
            warning_delay = round(max(0.0, first_detection_time - entered_warning_time), 3)

        summary_row.update({
            'total_frames': frame_count,
            'detection_frames': detection_count,
            'detection_rate': round(detection_rate, 4),
            'first_detection_distance_m': round(first_detection_distance, 3) if first_detection_distance else None,
            'warning_delay_s': warning_delay,
            'min_distance_m': round(min_distance, 3) if min_distance else None,
            'miss_rate': round(miss_rate, 4),
            'false_positive_count': false_positive_count,
            'critical_zone_frames': critical_zone_frames,
            'critical_zone_misses': critical_zone_misses,
        })

        print(f"  Risultato: detection_rate={detection_rate:.2%}, miss_rate={miss_rate:.2%}")

    except Exception as exc:
        summary_row['status'] = 'failed'
        print(f"  ERRORE: {exc}")

    finally:
        if camera:
            try:
                camera.stop()
                camera.destroy()
            except:
                pass
        if scenario:
            scenario.cleanup()

    return summary_row, frame_rows


# ---- Esegui Scenario 2 ----
print(f"Scenario 2 avviato | {len(TESTS_SCENARIO2)} condizioni")
print("-" * 60)

s2_summary_rows = []
s2_frame_rows = []

for idx, test in enumerate(TESTS_SCENARIO2, 1):
    print(f"\n[{idx}/{len(TESTS_SCENARIO2)}] Scenario 2 in corso...")
    summary_row, frame_rows = run_condition_crossing(
        world=world,
        model=model,
        batch_id=batch_id,
        weather=test['weather'],
        corruption=test['corruption'],
        severity=test['severity'],
    )
    s2_summary_rows.append(summary_row)
    s2_frame_rows.extend(frame_rows)

S2_FIELDS = [
    'batch_id', 'run_label', 'scenario', 'weather', 'corruption', 'severity',
    'total_frames', 'detection_frames', 'detection_rate',
    'first_detection_distance_m', 'warning_delay_s', 'min_distance_m',
    'miss_rate', 'false_positive_count', 'critical_zone_frames',
    'critical_zone_misses', 'status',
]
S2_FRAME_FIELDS = [
    'batch_id', 'run_label', 'scenario', 'weather', 'corruption', 'severity',
    'frame_index', 'elapsed_s', 'distance_m',
    'persons_detected', 'max_confidence', 'detected', 'in_critical_zone',
]
append_csv_rows(S2_SUMMARY_PATH, S2_FIELDS, s2_summary_rows)
append_csv_rows(S2_FRAME_PATH, S2_FRAME_FIELDS, s2_frame_rows)

print(f"\nScenario 2 completato!")
print(f"   Risultati: {S2_SUMMARY_PATH}")

## 10. Analisi Risultati

Carichiamo e visualizziamo i risultati del benchmark.

### 10.1 Aggregazione multi-batch

Se sono disponibili dati da più batch, li aggreghiamo per calcolare **media ± deviazione standard** per ogni condizione, rendendo i risultati statisticamente più robusti.

In [ ]:
# ============================================================
# AGGREGAZIONE MULTI-BATCH
# ============================================================

# Cerca tutti i run_summary.csv disponibili
summary_files = glob.glob('output_batch_*/results/run_summary.csv')
print(f"File run_summary trovati: {len(summary_files)}")

if summary_files:
    all_dfs = []
    for f in sorted(summary_files):
        try:
            df = pd.read_csv(f)
            all_dfs.append(df)
            print(f"   {f} ({len(df)} righe)")
        except Exception as e:
            print(f"   ERRORE {f}: {e}")
    
    if all_dfs:
        combined_df = pd.concat(all_dfs, ignore_index=True)
        print(f"\nTotale righe combinate: {len(combined_df)}")
        print(f"   Batch unici: {combined_df['batch_id'].nunique()}")
        
        # Converti colonne numeriche
        numeric_cols = ['detection_rate', 'first_detection_distance_m', 'min_distance_m',
                       'warning_delay_s', 'miss_rate', 'false_positive_count']
        for col in numeric_cols:
            if col in combined_df.columns:
                combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')
        
        # Aggrega per condizione (mean +/- std)
        agg_df = combined_df.groupby(['run_label', 'weather', 'corruption', 'severity']).agg(
            n_runs=('batch_id', 'nunique'),
            detection_rate_mean=('detection_rate', 'mean'),
            detection_rate_std=('detection_rate', 'std'),
            first_det_dist_mean=('first_detection_distance_m', 'mean'),
            first_det_dist_std=('first_detection_distance_m', 'std'),
            min_dist_mean=('min_distance_m', 'mean'),
            min_dist_std=('min_distance_m', 'std'),
        ).reset_index()
        
        agg_df = agg_df.fillna(0)
        
        # Label leggibile
        agg_df['condition'] = agg_df.apply(
            lambda r: f"{r['weather']} | {r['corruption']} | s{int(r['severity'])}",
            axis=1
        )
        
        print("\n" + "=" * 90)
        print("RISULTATI AGGREGATI (media +/- std)")
        print("=" * 90)
        
        for _, row in agg_df.sort_values(['weather', 'corruption', 'severity']).iterrows():
            dr = f"{row['detection_rate_mean']:.2%} +/- {row['detection_rate_std']:.2%}"
            dd = f"{row['first_det_dist_mean']:.1f}m +/- {row['first_det_dist_std']:.1f}m"
            print(f"  {row['condition']:30s} | det_rate={dr:20s} | first_dist={dd:20s} | n={int(row['n_runs'])}")
else:
    print("Nessun file run_summary trovato. Esegui il benchmark prima (Sezione 9).")
    agg_df = None
    combined_df = None

# ============================================================
# ULTIMO BATCH (per grafici dettagliati)
# ============================================================
if summary_files:
    latest_summary = sorted(summary_files)[-1]
    results_df = pd.read_csv(latest_summary)
    latest_batch = results_df['batch_id'].max()
    batch_df = results_df[results_df['batch_id'] == latest_batch].copy()
    
    batch_df['condition'] = batch_df.apply(
        lambda row: f"{row['weather']} | {row['corruption']} | s{int(row['severity'])}",
        axis=1
    )
    
    print(f"\nUltimo batch: {latest_batch} ({len(batch_df)} condizioni)")

## 11. Visualizzazione: Detection Rate per Condizione

Grafico a barre del detection rate per ogni condizione testata, con codifica colore per indicare le zone di performance:
- 🟢 Verde: buono (> 15%)
- 🟡 Arancione: degradato (8-15%)
- 🔴 Rosso: collasso (< 8%)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

if agg_df is not None and len(agg_df) > 0:
    plot_df = agg_df.sort_values('detection_rate_mean', ascending=False).copy()
    rate_col = 'detection_rate_mean'
    std_col = 'detection_rate_std'
else:
    plot_df = batch_df.sort_values('detection_rate', ascending=False).copy()
    rate_col = 'detection_rate'
    std_col = None

# Colori basati su performance
colors = []
for rate in plot_df[rate_col]:
    if rate > 0.15:
        colors.append('#2ecc71')  # Verde: buono
    elif rate > 0.08:
        colors.append('#f39c12')  # Arancione: degradato
    else:
        colors.append('#e74c3c')  # Rosso: collasso
        
bars = ax.bar(range(len(plot_df)), plot_df[rate_col], 
              color=colors, edgecolor='white', linewidth=0.5)

# Barre di errore (std) se disponibili
if std_col and std_col in plot_df.columns and plot_df[std_col].sum() > 0:
    ax.errorbar(range(len(plot_df)), plot_df[rate_col],
               yerr=plot_df[std_col], fmt='none', color='black',
               capsize=4, linewidth=1.5)

ax.set_xticks(range(len(plot_df)))
ax.set_xticklabels(plot_df['condition'], rotation=40, ha='right', fontsize=9)
ax.set_ylabel('Detection Rate', fontsize=12)
ax.set_xlabel('Condizione', fontsize=12)
ax.set_title('Robustezza Rilevamento Pedoni per Condizione Ambientale', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(plot_df[rate_col]) * 1.25)
ax.grid(axis='y', alpha=0.3)

# Soglia collasso
ax.axhline(y=0.05, color='red', linestyle='--', alpha=0.7, linewidth=1.5)

# Legenda colori
legend_elements = [
    mpatches.Patch(color='#2ecc71', label='Buono (>15%)'),
    mpatches.Patch(color='#f39c12', label='Degradato (8-15%)'),
    mpatches.Patch(color='#e74c3c', label='Collasso (<8%)'),
    plt.Line2D([0], [0], color='red', linestyle='--', label='Soglia collasso (5%)'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9)

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'detection_rate_by_condition.png')
plt.savefig(plot_path, dpi=160, bbox_inches='tight')
print(f"Grafico salvato: {plot_path}")
plt.show()

## 12. Grafico di Degradazione e Soglie di Collasso

Questo grafico mostra come il detection rate degrada al crescere della **severità** per ogni tipo di corruzione.

Il punto dove la curva scende sotto il **5%** indica la **soglia di collasso** del sistema.
La zona rossa evidenzia l'area di collasso.

In [ ]:
# ============================================================
# DEGRADAZIONE PROGRESSIVA PER TIPO DI CORRUZIONE
# ============================================================

if agg_df is not None and len(agg_df) > 0:
    source_df = agg_df.rename(columns={'detection_rate_mean': 'detection_rate'}).copy()
else:
    source_df = batch_df.copy()

fig, ax = plt.subplots(figsize=(12, 7))

# Per ogni tipo di corruzione, traccia detection_rate vs severity
corruption_types = source_df[source_df['corruption'] != 'none']['corruption'].unique()

# Punto base (clear, none, s0) come severity=0 per tutte le corruzioni
clear_baseline = source_df[(source_df['weather'] == 'clear') & (source_df['corruption'] == 'none')]
baseline_rate = clear_baseline['detection_rate'].values[0] if len(clear_baseline) > 0 else 0

markers = ['o', 's', '^', 'D', 'v']
colors_deg = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6', '#f39c12']

max_sev = 0
for i, corr_type in enumerate(sorted(corruption_types)):
    corr_data = source_df[(source_df['corruption'] == corr_type) & (source_df['weather'] == 'clear')]
    corr_data = corr_data.sort_values('severity')
    
    # Prepend baseline (severity=0)
    severities = [0] + list(corr_data['severity'].astype(int))
    rates = [baseline_rate] + list(corr_data['detection_rate'])
    max_sev = max(max_sev, max(severities))
    
    color = colors_deg[i % len(colors_deg)]
    marker = markers[i % len(markers)]
    
    ax.plot(severities, rates, marker=marker, color=color, linewidth=2.5,
            markersize=10, label=corr_type.replace('_', ' ').title(), zorder=3)

# Soglia collasso con area rossa
ax.axhline(y=0.05, color='red', linestyle='--', alpha=0.7, linewidth=2, label='Soglia collasso (5%)')
ax.fill_between([0, max(3, max_sev)], 0, 0.05, color='red', alpha=0.08)

ax.set_xlabel('Severita Corruzione', fontsize=13)
ax.set_ylabel('Detection Rate', fontsize=13)
ax.set_title('Degradazione Progressiva: Detection Rate vs Severita', fontsize=14, fontweight='bold')
ax.set_xticks(range(0, max_sev + 1))
ax.set_xticklabels(['0\n(none)'] + [f'{s}\n({"mild" if s==1 else "moderate" if s==2 else "severe"})' for s in range(1, max_sev + 1)])
ax.set_ylim(0, max(baseline_rate * 1.3, 0.25))
ax.legend(fontsize=11, loc='upper right')
ax.grid(alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'degradation_by_severity.png')
plt.savefig(plot_path, dpi=160, bbox_inches='tight')
print(f"Grafico degradazione salvato: {plot_path}")
plt.show()

## 13. Heatmap di Robustezza: Weather × Corruption

Matrice che mostra a colpo d'occhio quali combinazioni weather + corruption sono critiche per il sistema di detection.

Le celle più scure (rosse) indicano le combinazioni dove il sistema collassa.

In [ ]:
# ============================================================
# HEATMAP WEATHER x CORRUPTION
# ============================================================

if agg_df is not None and len(agg_df) > 0:
    heatmap_source = agg_df.rename(columns={'detection_rate_mean': 'detection_rate'}).copy()
else:
    heatmap_source = batch_df.copy()

# Crea label corruzione con severita
heatmap_source['corr_label'] = heatmap_source.apply(
    lambda r: f"{r['corruption']} s{int(r['severity'])}" if r['corruption'] != 'none' else 'none',
    axis=1
)

# Pivot table
pivot = heatmap_source.pivot_table(
    values='detection_rate', 
    index='weather', 
    columns='corr_label', 
    aggfunc='mean'
)

# Ordina
weather_order = ['clear', 'rain', 'fog', 'night', 'backlight']
pivot = pivot.reindex([w for w in weather_order if w in pivot.index])

fig, ax = plt.subplots(figsize=(14, 6))

sns.heatmap(pivot, annot=True, fmt='.2%', cmap='RdYlGn', ax=ax,
            linewidths=1, linecolor='white',
            cbar_kws={'label': 'Detection Rate'},
            vmin=0, vmax=max(0.20, pivot.max().max() * 1.1))

ax.set_title('Heatmap Robustezza: Detection Rate per Weather x Corruption', 
             fontsize=14, fontweight='bold')
ax.set_ylabel('Condizione Meteo', fontsize=12)
ax.set_xlabel('Corruzione Visiva', fontsize=12)

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'robustness_heatmap.png')
plt.savefig(plot_path, dpi=160, bbox_inches='tight')
print(f"Heatmap salvata: {plot_path}")
plt.show()

## 14. Confronto Side-by-Side per Condizione

Confronto visuale delle immagini annotate: un campione per ogni condizione meteo e corruzione testata.

Include almeno un confronto per: rain, fog, night, e blur (come richiesto dalla traccia).

In [ ]:
# Crea griglia con un sample per ogni condizione
sample_files = []
for _, row in batch_df.iterrows():
    condition_dir = os.path.join(SAMPLES_DIR, row['run_label'])
    if os.path.exists(condition_dir):
        all_samples = sorted([f for f in os.listdir(condition_dir) if f.endswith('.png')])
        for sample_file in all_samples:
            if 'detected' in sample_file or 'frame001' in sample_file:
                sample_path = os.path.join(condition_dir, sample_file)
                sample_files.append((row['condition'], sample_path))
                break

if sample_files:
    cols = 3
    rows = (len(sample_files) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes

    for idx, (label, sample_path) in enumerate(sample_files):
        img = Image.open(sample_path)
        axes[idx].imshow(img)
        axes[idx].set_title(label, fontsize=10, fontweight='bold')
        axes[idx].axis('off')

    for idx in range(len(sample_files), len(axes)):
        axes[idx].axis('off')

    plt.suptitle('Confronto Side-by-Side: Detection per Condizione', 
                 fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    
    comparison_path = os.path.join(PLOTS_DIR, 'side_by_side_comparison.png')
    plt.savefig(comparison_path, dpi=160, bbox_inches='tight')
    print(f"Comparison salvato: {comparison_path}")
    plt.show()
    
    print("\nSample disponibili per condizione:")
    for _, row in batch_df.iterrows():
        condition_dir = os.path.join(SAMPLES_DIR, row['run_label'])
        if os.path.exists(condition_dir):
            num_samples = len([f for f in os.listdir(condition_dir) if f.endswith('.png')])
            print(f"   {row['condition']}: {num_samples} immagini")
else:
    print("Nessun sample trovato per il confronto visuale")

## 15. Analisi Failure Cases

Identifichiamo e documentiamo i casi di fallimento specifici:
1. **Condizioni di collasso**: detection rate < 5% (il sistema non funziona)
2. **Condizioni degradate**: detection rate 5-10% (funzionamento parziale)
3. **Miss rate in zona critica**: percentuale di frame nella zona < 30m dove il pedone non viene rilevato
4. **Falsi positivi**: detection "person" quando il pedone è troppo lontano (> 60m)
5. **Frame MISS**: immagini salvate dove il pedone era vicino ma non rilevato

In [ ]:
# ============================================================
# FAILURE CASE ANALYSIS
# ============================================================

print("=" * 70)
print("  FAILURE CASE ANALYSIS")
print("=" * 70)

if agg_df is not None and len(agg_df) > 0:
    analysis_source = agg_df.rename(columns={'detection_rate_mean': 'detection_rate'}).copy()
else:
    analysis_source = batch_df.copy()

# 1. Condizioni di collasso (detection_rate < 5%)
collapsed = analysis_source[analysis_source['detection_rate'] < 0.05]
print("\nCONDIZIONI DI COLLASSO (detection rate < 5%):")
if len(collapsed) > 0:
    for _, row in collapsed.iterrows():
        print(f"   COLLASSO: {row['condition']}: {row['detection_rate']:.2%}")
else:
    print("   Nessuna condizione con collasso totale (< 5%)")

# 2. Condizioni degradate (detection_rate < 10%)
degraded = analysis_source[(analysis_source['detection_rate'] >= 0.05) & 
                           (analysis_source['detection_rate'] < 0.10)]
print("\nCONDIZIONI DEGRADATE (detection rate 5-10%):")
if len(degraded) > 0:
    for _, row in degraded.iterrows():
        print(f"   DEGRADATO: {row['condition']}: {row['detection_rate']:.2%}")
else:
    print("   Nessuna condizione in zona degradata")

# 3. Condizioni stabili (detection_rate >= 10%)
stable = analysis_source[analysis_source['detection_rate'] >= 0.10]
print("\nCONDIZIONI STABILI (detection rate >= 10%):")
for _, row in stable.iterrows():
    print(f"   OK: {row['condition']}: {row['detection_rate']:.2%}")

# 4. Miss rate e falsi positivi (se disponibili nel batch corrente)
if 'miss_rate' in batch_df.columns:
    print("\nMISS RATE IN ZONA CRITICA (<30m):")
    for _, row in batch_df.sort_values('miss_rate', ascending=False).iterrows():
        miss_rate_val = pd.to_numeric(row.get('miss_rate', 0), errors='coerce')
        if pd.notna(miss_rate_val):
            level = "ALTO" if miss_rate_val > 0.9 else "MEDIO" if miss_rate_val > 0.7 else "BASSO"
            print(f"   [{level}] {row['condition']}: miss_rate={miss_rate_val:.2%}")

if 'false_positive_count' in batch_df.columns:
    fp_total = pd.to_numeric(batch_df['false_positive_count'], errors='coerce').sum()
    print(f"\nFALSI POSITIVI TOTALI: {int(fp_total)}")
    if fp_total > 0:
        for _, row in batch_df.iterrows():
            fp = pd.to_numeric(row.get('false_positive_count', 0), errors='coerce')
            if pd.notna(fp) and fp > 0:
                print(f"   {row['condition']}: {int(fp)} falsi positivi")

# 5. Frame MISS con immagini salvate
print("\nFAILURE CASE SAMPLES:")
miss_samples = []
for _, row in batch_df.iterrows():
    condition_dir = os.path.join(SAMPLES_DIR, row['run_label'])
    if os.path.exists(condition_dir):
        miss_files = [f for f in os.listdir(condition_dir) if 'MISS' in f]
        if miss_files:
            miss_samples.append((row['condition'], len(miss_files), condition_dir))
            print(f"   {row['condition']}: {len(miss_files)} frame MISS in {condition_dir}")

if not miss_samples:
    print("   (Nessun frame MISS salvato - verranno generati al prossimo benchmark)")

# Visualizza failure cases se disponibili
if miss_samples:
    n_plots = min(len(miss_samples), 4)
    fig, axes = plt.subplots(1, n_plots, figsize=(16, 5))
    if n_plots == 1:
        axes = [axes]
    
    for idx, (label, count, cdir) in enumerate(miss_samples[:n_plots]):
        miss_files = sorted([f for f in os.listdir(cdir) if 'MISS' in f])
        if miss_files:
            img = Image.open(os.path.join(cdir, miss_files[0]))
            axes[idx].imshow(img)
            axes[idx].set_title(f"MISS: {label}", fontsize=9, color='red', fontweight='bold')
            axes[idx].axis('off')
    
    plt.suptitle('Failure Cases: Frame dove il pedone NON e stato rilevato', 
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    failure_path = os.path.join(PLOTS_DIR, 'failure_cases.png')
    plt.savefig(failure_path, dpi=160, bbox_inches='tight')
    print(f"\nFailure cases plot salvato: {failure_path}")
    plt.show()

## 16. Confidence over Distance

Grafico che mostra come la **confidenza** media del modello cambia con la **distanza** dal pedone, per le condizioni principali.

Questo rivela il comportamento del modello in modo più ricco rispetto a un singolo detection rate:
- A quale distanza il modello inizia a "vedere" il pedone
- Con quanta certezza rileva in base alla condizione
- Come si confrontano condizioni diverse nella "finestra di rilevamento" 

In [ ]:
# ============================================================
# CONFIDENCE OVER DISTANCE CURVE
# ============================================================

frame_files = sorted(glob.glob('output_batch_*/results/frame_metrics.csv'))

if frame_files:
    latest_frame_file = frame_files[-1]
    frame_df = pd.read_csv(latest_frame_file)
    
    # Converti tipi
    for col in ['distance_m', 'max_confidence', 'detected']:
        frame_df[col] = pd.to_numeric(frame_df[col], errors='coerce')
    
    # Crea bin di distanza (ogni 5 metri)
    frame_df['distance_bin'] = pd.cut(frame_df['distance_m'], bins=range(0, 85, 5), right=False)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
    
    # Condizioni da plottare
    conditions_to_plot = ['clear_none0', 'rain_none0', 'fog_none0', 'night_none0', 'clear_blur2']
    colors_conf = {
        'clear_none0': '#2ecc71', 
        'rain_none0': '#3498db', 
        'fog_none0': '#95a5a6', 
        'night_none0': '#2c3e50', 
        'clear_blur2': '#e74c3c'
    }
    
    for cond in conditions_to_plot:
        cond_data = frame_df[frame_df['run_label'] == cond]
        if len(cond_data) == 0:
            continue
        
        color = colors_conf.get(cond, '#7f8c8d')
        label = cond.replace('_', ' | ').replace('none0', 'none s0')
        
        # Plot 1: Confidenza media vs distanza (solo frame con detection)
        detected_data = cond_data[cond_data['detected'] == 1]
        if len(detected_data) > 0:
            binned = detected_data.groupby('distance_bin')['max_confidence'].mean()
            x_vals = [interval.mid for interval in binned.index if pd.notna(interval)]
            y_vals = [binned[interval] for interval in binned.index if pd.notna(interval)]
            ax1.plot(x_vals, y_vals, marker='o', color=color, linewidth=2, 
                    markersize=6, label=label, alpha=0.8)
        
        # Plot 2: Detection rate vs distanza
        binned_rate = cond_data.groupby('distance_bin')['detected'].mean()
        x_vals = [interval.mid for interval in binned_rate.index if pd.notna(interval)]
        y_vals = [binned_rate[interval] for interval in binned_rate.index if pd.notna(interval)]
        ax2.plot(x_vals, y_vals, marker='s', color=color, linewidth=2, 
                markersize=6, label=label, alpha=0.8)
    
    ax1.set_xlabel('Distanza dal pedone (m)', fontsize=12)
    ax1.set_ylabel('Confidenza media', fontsize=12)
    ax1.set_title('Confidenza Detection vs Distanza', fontsize=13, fontweight='bold')
    ax1.set_xlim(0, 80)
    ax1.set_ylim(0.5, 1.0)
    ax1.axhline(y=0.6, color='red', linestyle='--', alpha=0.5, label='Soglia conf (0.6)')
    ax1.legend(fontsize=9, loc='upper left')
    ax1.grid(alpha=0.3)
    ax1.invert_xaxis()
    
    ax2.set_xlabel('Distanza dal pedone (m)', fontsize=12)
    ax2.set_ylabel('Detection Rate', fontsize=12)
    ax2.set_title('Detection Rate vs Distanza', fontsize=13, fontweight='bold')
    ax2.set_xlim(0, 80)
    ax2.set_ylim(0, 1.05)
    ax2.legend(fontsize=9, loc='upper left')
    ax2.grid(alpha=0.3)
    ax2.invert_xaxis()
    
    plt.tight_layout()
    plot_path = os.path.join(PLOTS_DIR, 'confidence_over_distance.png')
    plt.savefig(plot_path, dpi=160, bbox_inches='tight')
    print(f"Confidence over distance plot salvato: {plot_path}")
    plt.show()
else:
    print("Nessun frame_metrics.csv trovato. Esegui il benchmark prima.")

## 17. Valutazione FCW (Forward Collision Warning)

Valutiamo le performance del modulo FCW per ogni condizione:
- **Precision**: % warning che sono corretti (vs falsi allarmi)  
- **Recall**: % pericoli reali che vengono segnalati (vs missed warnings)
- **Warning delay**: tempo di reazione dal momento in cui il pedone entra nella zona pericolosa

In [ ]:
# ============================================================
# VALUTAZIONE FCW
# ============================================================

if 'fcw_precision' in batch_df.columns:
    print("=" * 70)
    print("  FORWARD COLLISION WARNING - EVALUATION")
    print("=" * 70)
    
    fcw_cols = ['condition', 'fcw_precision', 'fcw_recall', 'fcw_true_positives',
                'fcw_missed_warnings', 'fcw_false_alarms', 'fcw_warning_delay_s']
    available_fcw_cols = [c for c in fcw_cols if c in batch_df.columns]
    
    print("\nMetriche FCW per condizione:")
    print(batch_df[available_fcw_cols].to_string(index=False))
    
    # Plot FCW precision/recall
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    fcw_df = batch_df.copy()
    for col in ['fcw_precision', 'fcw_recall', 'fcw_warning_delay_s']:
        if col in fcw_df.columns:
            fcw_df[col] = pd.to_numeric(fcw_df[col], errors='coerce').fillna(0)
    
    fcw_df = fcw_df.sort_values('fcw_recall', ascending=False)
    
    # Plot precision e recall
    x = range(len(fcw_df))
    width = 0.35
    ax1.bar([i - width/2 for i in x], fcw_df['fcw_precision'], width, 
            label='Precision', color='#3498db', alpha=0.8)
    ax1.bar([i + width/2 for i in x], fcw_df['fcw_recall'], width, 
            label='Recall', color='#e74c3c', alpha=0.8)
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(fcw_df['condition'], rotation=40, ha='right', fontsize=8)
    ax1.set_ylabel('Score', fontsize=12)
    ax1.set_title('FCW Precision & Recall per Condizione', fontsize=13, fontweight='bold')
    ax1.set_ylim(0, 1.05)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Plot warning delay
    if 'fcw_warning_delay_s' in fcw_df.columns:
        fcw_df_delay = fcw_df.sort_values('fcw_warning_delay_s', ascending=False)
        colors_delay = ['#e74c3c' if d > 1.0 else '#f39c12' if d > 0.5 else '#2ecc71' 
                       for d in fcw_df_delay['fcw_warning_delay_s']]
        ax2.barh(range(len(fcw_df_delay)), fcw_df_delay['fcw_warning_delay_s'], color=colors_delay)
        ax2.set_yticks(range(len(fcw_df_delay)))
        ax2.set_yticklabels(fcw_df_delay['condition'], fontsize=8)
        ax2.set_xlabel('Warning Delay (s)', fontsize=12)
        ax2.set_title('FCW Warning Delay per Condizione', fontsize=13, fontweight='bold')
        ax2.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plot_path = os.path.join(PLOTS_DIR, 'fcw_evaluation.png')
    plt.savefig(plot_path, dpi=160, bbox_inches='tight')
    print(f"\nFCW evaluation plot salvato: {plot_path}")
    plt.show()

else:
    print("Dati FCW non disponibili. Esegui il benchmark con il modulo FCW (Sezione 9).")
    print("   I dati FCW verranno generati automaticamente al prossimo run.")

## 18. Report Finale

Generiamo un report testuale completo con:
- Metodologia e configurazione
- Risultati aggregati (media ± std se multi-batch)
- Failure cases e condizioni di collasso
- Valutazione FCW
- Lista deliverables

In [ ]:
report_path = os.path.join(RESULTS_DIR, 'final_report.md')

# Prepara dati per il report
if agg_df is not None and len(agg_df) > 0:
    report_source = agg_df.rename(columns={'detection_rate_mean': 'detection_rate'}).copy()
    is_aggregated = True
else:
    report_source = batch_df.copy()
    is_aggregated = False

best_condition = report_source.loc[report_source['detection_rate'].idxmax()]
worst_condition = report_source.loc[report_source['detection_rate'].idxmin()]

report_lines = []
report_lines.append('# Project 11 - Adverse Weather Robustness Benchmark')
report_lines.append('')
report_lines.append('## Obiettivo')
report_lines.append('Valutare la robustezza del rilevamento pedoni (YOLOv8n) in CARLA sotto condizioni')
report_lines.append('ambientali avverse e corruzioni visive progressive, identificando soglie di collasso.')
report_lines.append('')
report_lines.append('## Metodologia')
report_lines.append('- **Baseline**: YOLOv8n pedestrian detection (soglia confidenza: 0.6)')
report_lines.append('- **Scenario**: pedone a distanza fissa (80m), traiettoria costante, velocita costante')
report_lines.append('- **Condizioni meteo**: clear, rain, fog, night, backlight')
report_lines.append('- **Corruzioni visive**: Gaussian blur (s1-s3), noise (s1-s3), motion blur (s1-s2), contrast (s1-s2)')
report_lines.append('- **Combinazioni**: rain+blur, fog+noise')
report_lines.append('- **Metriche**: detection rate, first detection distance, min distance, warning delay,')
report_lines.append('  miss rate, false positive count, FCW precision/recall')
report_lines.append(f'- **Test totali**: {len(TESTS)} condizioni')
if is_aggregated:
    report_lines.append(f'- **Batch aggregati**: {int(report_source["n_runs"].max())} run per condizione')
report_lines.append('')

report_lines.append('## Risultati')
report_lines.append('')
display_cols = ['condition', 'detection_rate']
if 'first_det_dist_mean' in report_source.columns:
    display_cols.append('first_det_dist_mean')
if 'min_dist_mean' in report_source.columns:
    display_cols.append('min_dist_mean')
available_cols = [c for c in display_cols if c in report_source.columns]
report_lines.append(report_source[available_cols].sort_values('detection_rate', ascending=False).to_markdown(index=False))
report_lines.append('')

report_lines.append('## Interpretazione')
report_lines.append(f'- **Migliore**: {best_condition["condition"]} ({best_condition["detection_rate"]:.2%})')
report_lines.append(f'- **Peggiore**: {worst_condition["condition"]} ({worst_condition["detection_rate"]:.2%})')
report_lines.append('')
report_lines.append('### Osservazioni principali')
report_lines.append('- La pioggia degrada significativamente il detection rate')
report_lines.append('- Il blur progressivo (s1 -> s2 -> s3) mostra degradazione chiara con soglia di collasso')
report_lines.append('- Il noise gaussiano ad alta severita compromette la detection')
report_lines.append('- Le combinazioni weather+corruption (es. rain+blur) sono le piu critiche')
report_lines.append('- Night ha impatto moderato grazie ai fari del veicolo')
report_lines.append('')

# Failure cases
report_lines.append('## Failure Cases')
collapsed = report_source[report_source['detection_rate'] < 0.05]
if len(collapsed) > 0:
    report_lines.append('')
    report_lines.append('### Condizioni di collasso (detection rate < 5%)')
    for _, row in collapsed.iterrows():
        report_lines.append(f'- **{row["condition"]}**: {row["detection_rate"]:.2%}')
else:
    report_lines.append('')
    report_lines.append('Nessuna condizione con collasso totale (< 5%).')

# FCW
report_lines.append('')
report_lines.append('## Forward Collision Warning (FCW)')
report_lines.append(f'- Soglia WARNING: {ForwardCollisionWarning.FCW_WARNING_DISTANCE}m')
report_lines.append(f'- Soglia DANGER: {ForwardCollisionWarning.FCW_DANGER_DISTANCE}m')
report_lines.append('- Il modulo FCW valuta precision, recall e warning delay per ogni condizione')

# Deliverables
report_lines.append('')
report_lines.append('## Deliverables')
report_lines.append('- Benchmark self-contained (notebook Jupyter)')
report_lines.append('- Scenario CARLA con gestione meteo')
report_lines.append('- Corruzioni visive multiple (blur, noise, motion blur, contrast)')
report_lines.append('- CSV con metriche per-frame e per-run')
report_lines.append('- Grafici: detection rate, degradazione, heatmap, confidence/distance, FCW')
report_lines.append('- Confronti side-by-side per tutte le condizioni')
report_lines.append('- Report con metodologia, metriche e failure cases')
report_lines.append('- Modulo FCW con precision/recall e warning delay')

with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_lines))

print(f"Report finale salvato: {report_path}")
print("\nContenuto report:")
print("=" * 80)
print('\n'.join(report_lines[:35]))
print("...")

## 20. Analisi Comparativa: Scenario 1 vs Scenario 2 ← NUOVO

Confrontiamo le performance del sistema nelle due configurazioni:
- **Scenario 1** (MovingScenario): pedone **fisso** a 80m, veicolo si avvicina
- **Scenario 2** (CrossingScenario): pedone **attraversa** la strada a 25m

Questo confronto mostra quanto la **dinamicità della scena** impatti la robustezza del detector.

In [ ]:
# ============================================================
# ANALISI COMPARATIVA SCENARIO 1 vs SCENARIO 2
# ============================================================

# Carica dati Scenario 1 (ultimo batch)
s1_df = pd.read_csv(sorted(glob.glob('output_batch_*/results/run_summary.csv'))[-1])
s1_latest = s1_df[s1_df['batch_id'] == s1_df['batch_id'].max()].copy()
s1_latest['scenario'] = 'S1: Pedone fisso'

# Carica dati Scenario 2
s2_df = pd.read_csv(S2_SUMMARY_PATH)
s2_df['scenario'] = 'S2: Pedone attraversa'

# Normalizza run_label per confronto (rimuovi prefisso s2_)
s2_df['run_label_base'] = s2_df['run_label'].str.replace('s2_', '', regex=False)
s1_latest['run_label_base'] = s1_latest['run_label']

# Condizioni comuni tra i due scenari
common_conditions = set(s2_df['run_label_base']) & set(s1_latest['run_label_base'])
s1_common = s1_latest[s1_latest['run_label_base'].isin(common_conditions)].copy()
s2_common = s2_df[s2_df['run_label_base'].isin(common_conditions)].copy()

# Label leggibile
for df in [s1_common, s2_common]:
    df['condition'] = df.apply(
        lambda r: f"{r['weather']} | {r['corruption']} | s{int(r['severity'])}",
        axis=1
    )

# ---- GRAFICO COMPARATIVO: Detection Rate ----
fig, ax = plt.subplots(figsize=(14, 7))

conditions_sorted = sorted(common_conditions)
x = np.arange(len(conditions_sorted))
width = 0.35

s1_rates = []
s2_rates = []
for cond in conditions_sorted:
    r1 = s1_common[s1_common['run_label_base'] == cond]['detection_rate'].values
    r2 = s2_common[s2_common['run_label_base'] == cond]['detection_rate'].values
    s1_rates.append(float(r1[0]) if len(r1) > 0 else 0)
    s2_rates.append(float(r2[0]) if len(r2) > 0 else 0)

bars1 = ax.bar(x - width/2, s1_rates, width, label='S1: Pedone fisso',
               color='#3498db', alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + width/2, s2_rates, width, label='S2: Pedone attraversa',
               color='#e74c3c', alpha=0.85, edgecolor='white')

# Differenza percentuale sopra le barre
for i, (r1, r2) in enumerate(zip(s1_rates, s2_rates)):
    if r1 > 0:
        diff = ((r2 - r1) / r1) * 100
        color = '#e74c3c' if diff < 0 else '#2ecc71'
        ax.text(i, max(r1, r2) + 0.005, f'{diff:+.0f}%',
                ha='center', va='bottom', fontsize=8, color=color, fontweight='bold')

ax.set_xticks(x)
labels_sorted = []
for cond in conditions_sorted:
    row = s1_common[s1_common['run_label_base'] == cond]
    if len(row) > 0:
        r = row.iloc[0]
        labels_sorted.append(f"{r['weather']} | {r['corruption']} s{int(r['severity'])}")
    else:
        labels_sorted.append(cond)
ax.set_xticklabels(labels_sorted, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Detection Rate', fontsize=12)
ax.set_title('Confronto Detection Rate: Pedone Fisso vs Pedone in Attraversamento',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, max(max(s1_rates), max(s2_rates)) * 1.30)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=0.05, color='gray', linestyle='--', alpha=0.5, label='Soglia collasso')

plt.tight_layout()
comp_path = os.path.join(PLOTS_DIR, 'scenario1_vs_scenario2.png')
plt.savefig(comp_path, dpi=160, bbox_inches='tight')
print(f"Grafico comparativo salvato: {comp_path}")
plt.show()

# ---- TABELLA COMPARATIVA ----
print("\n" + "=" * 80)
print("CONFRONTO SCENARIO 1 vs SCENARIO 2")
print("=" * 80)
print(f"{'Condizione':30s} | {'S1 det_rate':12s} | {'S2 det_rate':12s} | {'Delta':8s}")
print("-" * 70)
for cond, r1, r2 in zip(conditions_sorted, s1_rates, s2_rates):
    delta = r2 - r1
    delta_str = f"{delta:+.2%}"
    arrow = "↓" if delta < 0 else "↑"
    print(f"{cond:30s} | {r1:10.2%}   | {r2:10.2%}   | {arrow} {delta_str}")

# ---- INSIGHT ----
avg_delta = sum(r2 - r1 for r1, r2 in zip(s1_rates, s2_rates)) / len(s1_rates)
print(f"\nDelta medio: {avg_delta:+.2%}")
if avg_delta < 0:
    print(f"Il pedone in attraversamento e' piu' difficile da rilevare ({abs(avg_delta):.1%} in meno in media)")
else:
    print(f"Il pedone in attraversamento e' leggermente piu' facile da rilevare ({avg_delta:.1%} in piu' in media)")

## 19. Conclusioni e Next Steps

### Risultati ottenuti

Il benchmark ha dimostrato che:
1. **Rain** è la condizione meteo più critica per il detection
2. **Blur severo** (s3) e **noise** ad alta severità causano collasso del sistema
3. Le **combinazioni** weather + corruption (rain+blur, fog+noise) sono le più pericolose
4. **Night** e **backlight** hanno impatto moderato grazie ai fari del veicolo
5. Il modulo **FCW** mostra calo di recall significativo nelle condizioni degradate

### Confronto con la traccia del progetto

| Requisito | Stato |
|---|---|
| Baseline chiara (pedestrian detection) | ✅ YOLOv8n |
| CARLA weather parameters | ✅ 5 condizioni meteo |
| Image post-processing | ✅ 4 tipi di corruzione (blur, noise, motion blur, contrast) |
| Batch execution scripts | ✅ Notebook self-contained |
| Detection metrics | ✅ detection rate, first distance, min distance |
| Warning delay tracking | ✅ Ripristinato con zona critica |
| Error-rate tracking | ✅ miss rate + false positive count |
| Fixed scenarios (trajectory, speed, traffic constant) | ✅ MovingScenario con parametri fissi |
| Mild and severe visual corruptions | ✅ Severità 1-3 per ogni corruzione |
| Failure thresholds identification | ✅ Grafici di degradazione con soglia 5% |
| Tables and plots showing baseline collapse | ✅ Heatmap + degradation + failure analysis |
| Side-by-side comparison (rain, fog, night, blur) | ✅ Griglia per tutte le condizioni |
| Report with methodology, metrics, failure cases | ✅ Report finale completo |
| Forward Collision Warning module | ✅ FCW con precision/recall/delay |

### Possibili estensioni future

- Testare modelli più grandi (YOLOv8m, YOLOv8l) per confronto robustezza
- Aggiungere un modulo di lane detection e valutarne la degradazione
- Test con pedoni in movimento (attraversamento strada)
- Integrare metriche IoU con ground truth CARLA
- Eseguire test multipli per ogni condizione per statistiche più robuste

### File prodotti

```
output_batch_YYYYMMDD_HHMMSS/
├── results/
│   ├── run_summary.csv          # Metriche aggregate per condizione
│   ├── frame_metrics.csv        # Metriche per-frame
│   └── final_report.md          # Report completo
├── detection_samples/
│   ├── clear_none0/             # Sample annotati per condizione
│   ├── rain_none0/
│   ├── ...
│   └── fog_noise1/
└── plots/
    ├── detection_rate_by_condition.png
    ├── degradation_by_severity.png
    ├── robustness_heatmap.png
    ├── side_by_side_comparison.png
    ├── confidence_over_distance.png
    ├── fcw_evaluation.png
    └── failure_cases.png
```

---

**Fine del notebook**